# Análise de Fourier e Inicialização Guiada

Notebook refatorado para estudar a paisagem periódica do VQE sem ruído. O objetivo é mostrar quando uma aproximação de Fourier de baixa ordem ajuda a escolher um ponto inicial melhor para o otimizador.


## Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.fourier import (
    analyze_fourier_line,
    build_fourier_problem,
    estimate_first_harmonic_guided_point,
    make_coordinate_direction,
    run_budget_comparison,
    run_vqe_reference_point,
    scan_harmonic_error,
    scan_spectral_profile,
    spectral_metrics,
)
from src.vqe.ansatz import build_ansatz
from src.vqe.molecular_system import default_statevector_systems
from src.visualization.fourier_plots import (
    plot_budget_comparison,
    plot_fourier_reconstruction,
    plot_harmonic_error,
    plot_harmonic_profile,
    plot_spectral_metrics,
    save_figure,
)

pd.set_option("display.max_columns", None)
plt.style.use("dark_background")

output_dir = "outputs/figures/fourier"
seed = 137


## Sistemas e Configuração

Começamos com `H2` e `LiH` em `sto-3g`, usando poucos pontos de distância. Isso mantém o notebook executável para gerar figuras de slide.


In [ ]:
systems = [
    system
    for system in default_statevector_systems()
    if system.name in {"H2", "LiH"}
]

ansatz_name = "real_amplitudes"
reps = 2
mapper = "parity"
z2symmetry_reduction = True
optimizer_name = "cobyla"
reference_max_iter = 250
theta_samples = 64

[(system.name, system.basis, system.distances, system.active_space) for system in systems]


## Demonstração: uma linha Fourier em H2

Aqui escolhemos um ponto próximo ao ótimo VQE, variamos um parâmetro por vez e reconstruímos `E(theta)` com poucos harmônicos.


In [ ]:
demo_system = systems[0]
demo_distance = demo_system.distances[1]

problem, qubit_op, constant_energy = build_fourier_problem(
    demo_system,
    demo_distance,
    mapper=mapper,
    z2symmetry_reduction=z2symmetry_reduction,
)
ansatz = build_ansatz(
    name=ansatz_name,
    num_qubits=qubit_op.num_qubits,
    reps=reps,
    num_particles=problem.num_particles,
    num_spatial_orbitals=problem.num_spatial_orbitals,
)

vqe_reference = run_vqe_reference_point(
    qubit_op=qubit_op,
    ansatz=ansatz,
    constant_energy=constant_energy,
    optimizer_name=optimizer_name,
    max_iter=reference_max_iter,
    seed=seed,
)

center = np.asarray(vqe_reference["optimal_params"], dtype=float)
direction = make_coordinate_direction(ansatz.num_parameters, parameter_index=0)
line = analyze_fourier_line(
    ansatz=ansatz,
    qubit_op=qubit_op,
    constant_energy=constant_energy,
    center=center,
    direction=direction,
    theta_samples=theta_samples,
)

spectral_metrics(line.coefficients)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_fourier_reconstruction(line, harmonic_orders=(1, 2, 3), ax=ax)
path = save_figure(fig, output_dir, "fourier_reconstruction_h2.png")
print(path)
plt.show()


## Ponto Inicial Guiado

Com três avaliações em `theta = 0, pi/2, -pi/2`, estimamos o primeiro harmônico e calculamos o mínimo analítico dessa aproximação.


In [ ]:
guided_point, guide_cost, guide_info = estimate_first_harmonic_guided_point(
    ansatz=ansatz,
    qubit_op=qubit_op,
    constant_energy=constant_energy,
    center=center,
    direction=direction,
)

pd.DataFrame([{
    "molecule": demo_system.name,
    "basis": demo_system.basis,
    "distance": demo_distance,
    "guide_cost": guide_cost,
    **guide_info,
}])


## Perfil Espectral Local vs Global

`R1` mede quanto da energia espectral está no primeiro harmônico. `H_norm` mede quão espalhada está a energia entre harmônicos.


In [ ]:
RUN_SPECTRAL_SCAN = True

if RUN_SPECTRAL_SCAN:
    spectral_df, profile_df = scan_spectral_profile(
        systems=systems,
        ansatz_name=ansatz_name,
        reps=reps,
        mapper=mapper,
        z2symmetry_reduction=z2symmetry_reduction,
        optimizer_name=optimizer_name,
        max_iter=reference_max_iter,
        theta_samples=theta_samples,
        seed=seed,
        global_samples=3,
        max_harmonics=8,
    )

    display(spectral_df.head())
    display(profile_df.head())


In [ ]:
if RUN_SPECTRAL_SCAN:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_harmonic_profile(profile_df, ax=ax)
    path = save_figure(fig, output_dir, "fourier_harmonic_profile.png")
    print(path)
    plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    plot_spectral_metrics(spectral_df, ax=ax)
    path = save_figure(fig, output_dir, "fourier_spectral_metrics.png")
    print(path)
    plt.show()


## Erro vs Ordem Harmônica

Este bloco mostra se `K=1` já é suficiente ou se precisamos de harmônicos adicionais para reconstruir a curva e o mínimo.


In [ ]:
RUN_HARMONIC_ERROR = True

if RUN_HARMONIC_ERROR:
    harmonic_error_df = scan_harmonic_error(
        systems=systems,
        harmonic_grid=(1, 2, 3, 5),
        ansatz_name=ansatz_name,
        reps=reps,
        mapper=mapper,
        z2symmetry_reduction=z2symmetry_reduction,
        optimizer_name=optimizer_name,
        max_iter=reference_max_iter,
        theta_samples=theta_samples,
        seed=seed,
    )

    harmonic_summary = (
        harmonic_error_df.groupby(["molecule", "K"], as_index=False)
        .agg(mean_rmse=("rmse", "mean"), mean_delta_min=("delta_min_energy", "mean"))
        .sort_values(["molecule", "K"])
    )
    display(harmonic_summary)


In [ ]:
if RUN_HARMONIC_ERROR:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    plot_harmonic_error(harmonic_error_df, ax=ax)
    path = save_figure(fig, output_dir, "fourier_error_vs_k.png")
    print(path)
    plt.show()


## Inicialização Aleatória vs Fourier-Guided

A hipótese principal: a inicialização guiada melhora um mesmo ponto inicial aleatório, especialmente quando o orçamento de iterações é pequeno.


In [ ]:
RUN_BUDGET_COMPARISON = True

if RUN_BUDGET_COMPARISON:
    budget_df = run_budget_comparison(
        systems=systems,
        iteration_grid=(25, 50, 100, 200),
        ansatz_name=ansatz_name,
        reps=reps,
        mapper=mapper,
        z2symmetry_reduction=z2symmetry_reduction,
        optimizer_name=optimizer_name,
        seed=seed,
        repeats=3,
    )

    budget_summary = (
        budget_df.groupby(["max_iter", "molecule", "mode"], as_index=False)
        .agg(mean_total_cost=("total_cost", "mean"), mean_abs_error=("abs_error", "mean"))
        .sort_values(["molecule", "max_iter", "mode"])
    )
    display(budget_summary)


In [ ]:
if RUN_BUDGET_COMPARISON:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    plot_budget_comparison(budget_df, ax=ax)
    path = save_figure(fig, output_dir, "fourier_budget_comparison.png")
    print(path)
    plt.show()


## Dados Para Slides


In [ ]:
for name in ["spectral_df", "profile_df", "harmonic_error_df", "budget_df"]:
    if name in globals():
        print(name, globals()[name].shape)

# Exemplo de tabela compacta para copiar para slide/texto.
if "budget_df" in globals():
    display(
        budget_df.groupby(["max_iter", "molecule", "mode"], as_index=False)
        .agg(
            mean_error=("abs_error", "mean"),
            median_error=("abs_error", "median"),
            mean_total_cost=("total_cost", "mean"),
        )
        .sort_values(["molecule", "max_iter", "mode"])
    )


### Relevância da inicialização vs orçamento
Esta célula resume a ideia central: a inicialização guiada tende a importar mais quando o otimizador tem poucas iterações.
Com mais iterações, a diferença entre partir de um ponto aleatório e partir do ponto Fourier-guided deve diminuir.

In [ ]:
if "budget_df" not in globals():
    raise RuntimeError("Execute primeiro a seção 'Inicialização Aleatória vs Fourier-Guided'.")

iteration_effect = (
    budget_df.groupby(["max_iter", "molecule", "mode"], as_index=False)
    .agg(
        mean_abs_error=("abs_error", "mean"),
        median_abs_error=("abs_error", "median"),
        mean_total_cost=("total_cost", "mean"),
    )
)

iteration_wide = iteration_effect.pivot_table(
    index=["max_iter", "molecule"],
    columns="mode",
    values=["mean_abs_error", "median_abs_error", "mean_total_cost"],
    aggfunc="first",
).reset_index()
iteration_wide.columns = [
    "max_iter",
    "molecule",
    "mean_error_guided",
    "mean_error_random",
    "median_error_guided",
    "median_error_random",
    "mean_cost_guided",
    "mean_cost_random",
]

iteration_wide["absolute_error_gain"] = (
    iteration_wide["mean_error_random"] - iteration_wide["mean_error_guided"]
)
iteration_wide["relative_error_gain"] = 1.0 - (
    iteration_wide["mean_error_guided"] / (iteration_wide["mean_error_random"] + 1e-16)
)
iteration_wide["cost_delta"] = (
    iteration_wide["mean_cost_guided"] - iteration_wide["mean_cost_random"]
)

display(iteration_wide.sort_values(["molecule", "max_iter"]).round(8))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for molecule, group in iteration_wide.groupby("molecule"):
    group = group.sort_values("max_iter")
    ax[0].plot(
        group["max_iter"],
        group["absolute_error_gain"],
        marker="o",
        linewidth=2,
        label=molecule,
    )
    ax[1].plot(
        group["max_iter"],
        group["relative_error_gain"],
        marker="o",
        linewidth=2,
        label=molecule,
    )

ax[0].axhline(0.0, color="#888888", linewidth=1)
ax[0].set_title("Ganho absoluto do Fourier-guided")
ax[0].set_xlabel("Máximo de iterações")
ax[0].set_ylabel("erro_random - erro_guided")
ax[0].grid(alpha=0.25)

ax[1].axhline(0.0, color="#888888", linewidth=1)
ax[1].set_title("Ganho relativo do Fourier-guided")
ax[1].set_xlabel("Máximo de iterações")
ax[1].set_ylabel("1 - erro_guided / erro_random")
ax[1].grid(alpha=0.25)
ax[1].legend()

path = save_figure(fig, output_dir, "fourier_initialization_relevance_vs_iterations.png")
print(path)
plt.show()
